# 04.1 â€” RAG + Context Reranking (Groq API)

Versi notebook 04 yang menggunakan **Groq API** (Llama 3.3-70B) sebagai pengganti Ollama lokal.

**Pipeline:**
```
Query â†’ BM25 (top-20 kandidat) â†’ CrossEncoder Rerank â†’ top-5 â†’ LLM Generate â†’ Extract Label
```

**Perubahan dari 04:**
- LLM: `llama-3.3-70b-versatile` via Groq (vs `llama3.2` via Ollama)
- Rate limiting otomatis (2.5s antar request)

**Yang sama:** BM25 index, CrossEncoder reranker, prompt generation, custom evaluator, format output JSON

In [ ]:
# Install dependencies (jalankan sekali)
# !pip install groq rank-bm25 datasets sentence-transformers

In [ ]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime

from groq import Groq
from rank_bm25 import BM25Okapi
from datasets import load_dataset
from sentence_transformers import CrossEncoder

warnings.filterwarnings('ignore')
print('Semua library berhasil diimpor!')
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | Pandas: {pd.__version__}')

In [ ]:
# ============================================================
# KONFIGURASI
# ============================================================

GROQ_API_KEY = os.environ.get('GROQ_API_KEY', 'YOUR_API_KEY_HERE')

LLM_MODEL      = 'llama-3.3-70b-versatile'
RERANKER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

TOP_K_CANDIDATES = 20   # BM25 ambil 20 kandidat
TOP_K_RETRIEVAL  = 5    # Setelah rerank, ambil top-5

DATASET_NAME   = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_labeled'
MAX_SAMPLES    = 500

TEMPERATURE = 0.0
SEED        = 42

NOTEBOOK_DIR    = Path('.')
BM25_INDEX_PATH = NOTEBOOK_DIR / 'pubmedqa_bm25.pkl'
RESULTS_DIR     = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

CONFIG_NAME        = 'cr_groq'
PHASE1_PATH        = RESULTS_DIR / f'{CONFIG_NAME}_phase1_answers.json'
PHASE2_CUSTOM_PATH = RESULTS_DIR / f'{CONFIG_NAME}_phase2_custom.json'

# Path hasil konfigurasi lain (untuk perbandingan)
BASELINE_GROQ_P1   = RESULTS_DIR / 'baseline_groq_phase1_answers.json'
BASELINE_GROQ_P2   = RESULTS_DIR / 'baseline_groq_phase2_custom.json'
QR_GROQ_P1         = RESULTS_DIR / 'qr_groq_phase1_answers.json'
QR_GROQ_P2         = RESULTS_DIR / 'qr_groq_phase2_custom.json'
CR_OLLAMA_P1        = RESULTS_DIR / 'cr_phase1_answers.json'
CR_OLLAMA_P2        = RESULTS_DIR / 'cr_phase2_custom.json'

print('Konfigurasi:')
print(f'  LLM          : {LLM_MODEL} (via Groq API)')
print(f'  Reranker     : {RERANKER_MODEL}')
print(f'  Retriever    : BM25 top-{TOP_K_CANDIDATES} -> CrossEncoder -> top-{TOP_K_RETRIEVAL}')
print(f'  Max Sampel   : {MAX_SAMPLES}')
print(f'  Config       : {CONFIG_NAME}')
print()
if 'YOUR_API_KEY' in GROQ_API_KEY:
    print('WARNING: GROQ_API_KEY belum diisi!')
else:
    print(f'GROQ_API_KEY: {GROQ_API_KEY[:8]}...{GROQ_API_KEY[-4:]}')

In [ ]:
@dataclass
class Document:
    text         : str
    pubid        : str
    question     : str
    section_label: str
    answer       : str
    decision     : str

@dataclass
class RetrievalResult:
    document      : Document
    score         : float           # BM25 score (kandidat awal)
    reranker_score: float = 0.0     # CrossEncoder score (setelah rerank)


def tokenize_bm25(text: str) -> List[str]:
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).split()

print('Data classes dan tokenizer siap.')

In [ ]:
def load_pubmedqa(subset=DATASET_SUBSET, max_samples=MAX_SAMPLES):
    print(f'Memuat PubMedQA ({subset})...')
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    data    = dataset['train']
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    print(f'Dimuat {len(data)} sampel')
    return data


def prepare_documents(data) -> List[Document]:
    docs = []
    for item in data:
        pubid = str(item['pubid'])
        for ctx, label in zip(item['context']['contexts'], item['context']['labels']):
            docs.append(Document(
                text=ctx.strip(), pubid=pubid,
                question=item['question'], section_label=label,
                answer=item['long_answer'], decision=item['final_decision']
            ))
    return docs


def load_or_build_bm25(data) -> Tuple[BM25Okapi, List[Document]]:
    if BM25_INDEX_PATH.exists():
        print(f'Memuat BM25 index dari {BM25_INDEX_PATH}...')
        with open(BM25_INDEX_PATH, 'rb') as f:
            saved = pickle.load(f)
        print(f'Dimuat: {len(saved["documents"])} dokumen')
        return saved['bm25'], saved['documents']
    else:
        print('Membangun BM25 index baru...')
        documents = prepare_documents(data)
        tokenized = [tokenize_bm25(d.text) for d in documents]
        bm25      = BM25Okapi(tokenized)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({'bm25': bm25, 'documents': documents}, f)
        return bm25, documents


pubmedqa_data         = load_pubmedqa()
bm25_index, documents = load_or_build_bm25(pubmedqa_data)
print(f'Evaluasi: {len(pubmedqa_data)} sampel, {len(documents)} dokumen.')

## CrossEncoder Reranker

**Model:** `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Dilatih pada MS MARCO passage ranking task
- Input: (query, passage) pair â†’ score relevance
- Ringan (~85MB), berjalan di CPU
- Digunakan untuk reranking top-20 kandidat BM25 menjadi top-5

In [ ]:
print(f'Memuat CrossEncoder: {RERANKER_MODEL}...')
print('(Download ~85MB sekali, lalu di-cache)')
t0 = time.time()
cross_encoder = CrossEncoder(RERANKER_MODEL)
print(f'CrossEncoder siap dalam {time.time()-t0:.1f} detik')

# Smoke test
_pairs = [
    ('Does aspirin prevent heart attacks?', 'Aspirin reduces platelet aggregation and is used in cardiovascular prevention.'),
    ('Does aspirin prevent heart attacks?', 'Weather patterns affect agricultural yields in tropical regions.'),
]
_scores = cross_encoder.predict(_pairs)
print(f'\nSmoke test CrossEncoder:')
print(f'  Relevan      : {_scores[0]:.4f}')
print(f'  Tidak relevan: {_scores[1]:.4f}')
assert _scores[0] > _scores[1], 'CrossEncoder gagal membedakan relevan vs tidak!'
print('Reranker berfungsi dengan benar.')

In [ ]:
# ============================================================
# Groq Client + Rate Limiting
# ============================================================

groq_client = Groq(api_key=GROQ_API_KEY)

GROQ_DELAY = 2.5
_last_call_time = 0


def groq_generate(prompt: str, max_tokens: int = 300, temperature: float = TEMPERATURE) -> str:
    global _last_call_time
    elapsed = time.time() - _last_call_time
    if elapsed < GROQ_DELAY:
        time.sleep(GROQ_DELAY - elapsed)

    for attempt in range(5):
        try:
            _last_call_time = time.time()
            response = groq_client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                wait = (attempt + 1) * 15
                print(f'  [Rate limit] Tunggu {wait}s... (attempt {attempt+1}/5)')
                time.sleep(wait)
            else:
                print(f'  [Groq Error] {type(e).__name__}: {err[:80]}')
                raise
    raise RuntimeError('Groq API gagal setelah 5 percobaan.')


print('Testing Groq API...')
_test = groq_generate('Reply with exactly: OK', max_tokens=5)
print(f'Response: {_test!r}')
print(f'Delay: {GROQ_DELAY}s | Groq client siap!')

## Retrieval dengan Context Reranking (BM25 -> CrossEncoder)

Dua tahap:
1. **BM25** mengambil top-20 kandidat (keyword matching)
2. **CrossEncoder** mereranking 20 kandidat tersebut berdasarkan semantic relevance, ambil top-5

In [ ]:
def retrieve_with_reranking(
    query: str,
    k_candidates: int = TOP_K_CANDIDATES,
    k_final: int = TOP_K_RETRIEVAL
) -> List[RetrievalResult]:
    """
    Retrieval dua tahap:
      1. BM25: ambil top-k_candidates (20) dokumen kandidat
      2. CrossEncoder: rerank kandidat, ambil top-k_final (5)
    """
    # Tahap 1: BM25 retrieval
    tokens    = tokenize_bm25(query)
    scores    = bm25_index.get_scores(tokens)
    top_cands = np.argsort(scores)[::-1][:k_candidates]
    candidates = [
        RetrievalResult(document=documents[i], score=float(scores[i]))
        for i in top_cands
    ]

    # Tahap 2: CrossEncoder reranking
    pairs = [(query, r.document.text) for r in candidates]
    reranker_scores = cross_encoder.predict(pairs)

    for r, rs in zip(candidates, reranker_scores):
        r.reranker_score = float(rs)

    # Sort by reranker score, ambil top-k_final
    reranked = sorted(candidates, key=lambda r: r.reranker_score, reverse=True)
    return reranked[:k_final]


# Test
test_q = 'Does aspirin reduce the risk of myocardial infarction?'
test_r = retrieve_with_reranking(test_q)
print(f'Query: {test_q}')
print(f'\nTop-{TOP_K_RETRIEVAL} dokumen (BM25 top-{TOP_K_CANDIDATES} -> CrossEncoder):')
for i, r in enumerate(test_r, 1):
    print(f'  [{i}] BM25={r.score:.2f} | Reranker={r.reranker_score:.4f} | {r.document.section_label} | {r.document.text[:80]}...')

In [ ]:
# Prompt identik dengan baseline â€” isolasi variabel: hanya retrieval yang berubah
GENERATION_PROMPT = (
    'You are a medical research assistant. '
    'Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n'
    'Context from medical literature:\n{context}\n\n'
    'Question: {question}\n\n'
    'Instructions:\n'
    '- Carefully read the context and assess whether it supports or refutes the question.\n'
    '- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n'
    '- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n'
    '  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n'
    '  - no    : the evidence refutes or does not support the hypothesis\n'
    '  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n'
    '            others say no), or if the context contains no relevant information at all\n'
    '- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n'
    '  Do NOT use maybe simply because the evidence is limited or not 100%% certain.\n\n'
    'Answer:'
)


def generate_answer(query: str, retrieved: List[RetrievalResult]) -> str:
    context = '\n\n'.join(
        f'[{i}] ({r.document.section_label}): {r.document.text}'
        for i, r in enumerate(retrieved, 1)
    )
    return groq_generate(
        GENERATION_PROMPT.format(context=context, question=query),
        max_tokens=300,
        temperature=TEMPERATURE
    )

print('Generation function siap.')

In [ ]:
def extract_label(answer: str) -> str:
    lines = [l.strip().lower() for l in answer.split('\n') if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r'[^a-z]', '', line)
        if word in ('yes', 'no', 'maybe'):
            return word
    for label in ('yes', 'no', 'maybe'):
        if re.search(r'\b' + label + r'\b', answer.lower()):
            return label
    return 'maybe'

print('extract_label siap.')

In [ ]:
# ============================================================
# Custom Zero-NaN Evaluator (via Groq)
# ============================================================

def _split_sentences(text: str) -> List[str]:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if len(s.strip()) >= 15]


def _llm_yes_no(prompt: str) -> bool:
    try:
        resp = groq_generate(prompt, max_tokens=10, temperature=0.0)
        return 'yes' in resp.lower()[:15]
    except Exception:
        return False


def compute_faithfulness(answer: str, contexts: List[str]) -> float:
    sentences = _split_sentences(answer)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement directly supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    supported = sum(
        1 for s in sentences
        if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s))
    )
    return supported / len(sentences)


def compute_context_recall(reference: str, contexts: List[str]) -> float:
    sentences = _split_sentences(reference)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    covered = sum(
        1 for s in sentences
        if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s))
    )
    return covered / len(sentences)


def evaluate_custom(question: str, answer: str,
                    contexts: List[str], reference: str) -> Dict:
    return {
        'faithfulness'  : compute_faithfulness(answer, contexts),
        'context_recall': compute_context_recall(reference, contexts),
    }

print('Zero-NaN evaluator siap (via Groq).')

## Demo â€” 5 Sampel

In [ ]:
DEMO_SIZE = 5

print(f'DEMO: {DEMO_SIZE} sampel (Groq CR: {LLM_MODEL})')
print('=' * 70)

for i in range(DEMO_SIZE):
    s          = pubmedqa_data[i]
    q, gt, ref = s['question'], s['final_decision'], s['long_answer']

    retrieved = retrieve_with_reranking(q)
    answer    = generate_answer(q, retrieved)
    predicted = extract_label(answer)

    status = 'BENAR' if predicted == gt else 'SALAH'
    print(f'\n[{i+1}/{DEMO_SIZE}] {q[:70]}...')
    print(f'  GT={gt} | Pred={predicted} | {status}')
    print(f'  Reranker scores: {[f"{r.reranker_score:.3f}" for r in retrieved]}')
    print(f'  Jawaban: {answer[:100]}...')

## Phase 1 â€” Generate Jawaban (500 Sampel)

Setiap sampel: BM25 top-20 -> CrossEncoder rerank -> top-5 -> 1 API call (generate).
Estimasi: ~21 menit untuk 500 sampel (2.5s delay x 500).

In [ ]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
        phase1_results = json.load(f)['results']
    start_from = len(phase1_results)
    print(f'Resume Phase 1: {start_from}/{MAX_SAMPLES} sudah selesai.')
else:
    phase1_results, start_from = [], 0
    print(f'Mulai Phase 1: {MAX_SAMPLES} sampel (BM25 top-{TOP_K_CANDIDATES} -> CR -> top-{TOP_K_RETRIEVAL}).')

if start_from < MAX_SAMPLES:
    print(f'Memproses {MAX_SAMPLES - start_from} sampel tersisa...\n')
    t_start = time.time()

    for i in range(start_from, MAX_SAMPLES):
        s          = pubmedqa_data[i]
        q, gt, ref = s['question'], s['final_decision'], s['long_answer']

        retrieved = retrieve_with_reranking(q)
        answer    = generate_answer(q, retrieved)
        predicted = extract_label(answer)

        phase1_results.append({
            'idx'             : i,
            'pubid'           : str(s['pubid']),
            'question'        : q,
            'ground_truth'    : gt,
            'predicted_label' : predicted,
            'is_correct'      : predicted == gt,
            'answer'          : answer,
            'contexts'        : [r.document.text for r in retrieved],
            'reference'       : ref,
            'retrieval_scores': [r.score for r in retrieved],
            'reranker_scores' : [r.reranker_score for r in retrieved],
        })

        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, 'w', encoding='utf-8') as f:
                json.dump({
                    'config'    : CONFIG_NAME,
                    'llm_model' : LLM_MODEL,
                    'reranker'  : RERANKER_MODEL,
                    'timestamp' : datetime.now().isoformat(),
                    'max_samples': MAX_SAMPLES,
                    'completed' : i + 1,
                    'results'   : phase1_results
                }, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc  = sum(r['is_correct'] for r in phase1_results) / done
            eta  = (time.time()-t_start) / done * (MAX_SAMPLES-done) / 60
            print(f'  [{done:3d}/{MAX_SAMPLES}] Accuracy: {acc:.1%} | pred={predicted}, gt={gt} | ETA {eta:.1f} mnt')

    print(f'\nPhase 1 selesai! -> {PHASE1_PATH}')
else:
    print(f'Phase 1 sudah selesai ({MAX_SAMPLES} sampel).')

## Analisis Phase 1

In [ ]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']

n         = len(results_p1)
n_correct = sum(r['is_correct'] for r in results_p1)
gts       = [r['ground_truth']    for r in results_p1]
preds     = [r['predicted_label'] for r in results_p1]

print(f'ANALISIS PHASE 1 -- {n} sampel ({CONFIG_NAME})')
print('=' * 55)
print(f'Label Accuracy    : {n_correct}/{n} = {n_correct/n:.1%}')
print(f'Hallucination Rate: {(n-n_correct)/n:.1%}\n')

print(f'  {"Label":<8} | {"Ground Truth":>12} | {"Prediksi":>10}')
print(f'  {"-"*40}')
for lbl in ['yes','no','maybe']:
    g, p = gts.count(lbl), preds.count(lbl)
    print(f'  {lbl:<8} | {g:>6} ({g/n:.0%})    | {p:>6} ({p/n:.0%})')

print('\nConfusion Matrix (baris=GT, kolom=Pred):')
lbls = ['yes','no','maybe']
print('  ' + f'{"GT/Pred":>8}' + ''.join(f'{l:>8}' for l in lbls))
for gt_l in lbls:
    row = f'  {gt_l:>8}'
    for pr_l in lbls:
        cnt = sum(1 for r in results_p1 if r['ground_truth']==gt_l and r['predicted_label']==pr_l)
        row += f'{cnt:>8}'
    print(row)

# Bandingkan dengan CR Ollama
if CR_OLLAMA_P1.exists():
    with open(CR_OLLAMA_P1) as f:
        cr_ol = json.load(f)['results'][:n]
    ol_acc = sum(r['is_correct'] for r in cr_ol) / len(cr_ol)
    print(f'\n--- Perbandingan CR ---')
    print(f'  CR Ollama (llama3.2)     : {ol_acc:.1%}')
    print(f'  CR Groq   (llama3.3-70b) : {n_correct/n:.1%}')
    print(f'  Delta                    : {(n_correct/n - ol_acc):+.1%}')

# Bandingkan dengan Baseline Groq
if BASELINE_GROQ_P1.exists():
    with open(BASELINE_GROQ_P1) as f:
        bl_g = json.load(f)['results'][:n]
    bl_acc = sum(r['is_correct'] for r in bl_g) / len(bl_g)
    print(f'\n--- CR vs Baseline (Groq) ---')
    print(f'  Baseline Groq: {bl_acc:.1%}')
    print(f'  CR Groq      : {n_correct/n:.1%}')
    print(f'  Delta        : {(n_correct/n - bl_acc):+.1%}')

## Phase 2 â€” Custom Evaluator (Faithfulness + Context Recall)

In [ ]:
MAX_CUSTOM_SAMPLES = MAX_SAMPLES

with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    p1_custom = json.load(f)['results'][:MAX_CUSTOM_SAMPLES]

if PHASE2_CUSTOM_PATH.exists():
    with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
        p2_custom = json.load(f)['results']
    done_custom = {r['idx'] for r in p2_custom}
    print(f'Resume Phase 2: {len(done_custom)}/{MAX_CUSTOM_SAMPLES} selesai.')
else:
    p2_custom, done_custom = [], set()
    print(f'Mulai Phase 2: {MAX_CUSTOM_SAMPLES} sampel (custom zero-NaN).')

remaining = [r for r in p1_custom if r['idx'] not in done_custom]
print(f'Sisa: {len(remaining)} sampel\n')

t0 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_custom(r['question'], r['answer'], r['contexts'], r['reference'])
    p2_custom.append({
        'idx'            : r['idx'],
        'ground_truth'   : r['ground_truth'],
        'predicted_label': r['predicted_label'],
        'is_correct'     : r['is_correct'],
        **scores
    })

    if (i + 1) % 5 == 0 or i == len(remaining) - 1:
        with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
            json.dump({
                'config'    : CONFIG_NAME,
                'llm_model' : LLM_MODEL,
                'reranker'  : RERANKER_MODEL,
                'timestamp' : datetime.now().isoformat(),
                'max_samples': MAX_CUSTOM_SAMPLES,
                'metrics'   : ['faithfulness', 'context_recall'],
                'evaluator' : 'custom_zero_nan',
                'results'   : p2_custom
            }, f, indent=2, ensure_ascii=False)
        done  = i + 1
        total = len(remaining)
        eta   = (time.time()-t0)/done*(total-done)/60 if done < total else 0
        avg_f = sum(x['faithfulness']   for x in p2_custom) / len(p2_custom)
        avg_r = sum(x['context_recall'] for x in p2_custom) / len(p2_custom)
        print(f'  [{done:3d}/{total}] idx={r["idx"]} | '
              f'faith={scores["faithfulness"]:.3f} | cr={scores["context_recall"]:.3f} | '
              f'avg_f={avg_f:.3f} | avg_cr={avg_r:.3f} | ETA {eta:.1f} mnt')

print(f'\nPhase 2 selesai! -> {PHASE2_CUSTOM_PATH}')

## Summary

In [ ]:
with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
    p2 = json.load(f)['results']

n      = len(p2)
acc    = sum(r['is_correct']     for r in p2) / n
avg_f  = sum(r['faithfulness']   for r in p2) / n
avg_cr = sum(r['context_recall'] for r in p2) / n

print('=' * 65)
print(f'  {CONFIG_NAME.upper()} -- {n} sampel')
print(f'  LLM: {LLM_MODEL} | Reranker: {RERANKER_MODEL}')
print('=' * 65)
print(f'  Label Accuracy    : {acc:.1%}')
print(f'  Hallucination Rate: {1-acc:.1%}')
print(f'  Faithfulness      : {avg_f:.4f}')
print(f'  Context Recall    : {avg_cr:.4f}')
print('=' * 65)

# Per-label
print('\nPer-label accuracy:')
for lbl in ['yes','no','maybe']:
    sub = [r for r in p2 if r['ground_truth'] == lbl]
    if sub:
        lbl_acc = sum(r['is_correct'] for r in sub) / len(sub)
        lbl_f   = sum(r['faithfulness'] for r in sub) / len(sub)
        lbl_cr  = sum(r['context_recall'] for r in sub) / len(sub)
        print(f'  {lbl:>5}: acc={lbl_acc:.1%} (n={len(sub)}) | faith={lbl_f:.3f} | ctx_recall={lbl_cr:.3f}')

# Perbandingan semua konfigurasi Groq
print('\n--- Perbandingan Semua Konfigurasi Groq ---')
configs = {
    'Baseline Groq': str(BASELINE_GROQ_P2),
    'QR Groq'      : str(QR_GROQ_P2),
    'CR Groq'      : str(PHASE2_CUSTOM_PATH),
}
for name, path in configs.items():
    p = Path(path)
    if p.exists():
        with open(p) as f:
            data = json.load(f)['results']
        a  = sum(r['is_correct'] for r in data) / len(data)
        ff = sum(r['faithfulness'] for r in data) / len(data)
        cr = sum(r['context_recall'] for r in data) / len(data)
        print(f'  {name:<16}: acc={a:.1%} | faith={ff:.4f} | ctx_recall={cr:.4f} (n={len(data)})')
    else:
        print(f'  {name:<16}: belum tersedia')

print(f'\nBaris tabel skripsi:')
print(f'  | CR Groq (llama-3.3-70b) | {acc:.3f} | {1-acc:.3f} | {avg_f:.3f} | {avg_cr:.3f} |')

## Analisis Kualitas Reranking

Inspeksi: seberapa besar CrossEncoder mengubah urutan dari BM25?

In [ ]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    cr_p1 = json.load(f)['results']

rr_spreads = []
for r in cr_p1:
    rr = r.get('reranker_scores', [])
    if len(rr) >= 2:
        rr_spreads.append(max(rr) - min(rr))

print('Analisis Reranker Score (top-5 per sampel):')
print('=' * 50)
all_rr = [s for r in cr_p1 for s in r.get('reranker_scores', [])]
if all_rr:
    print(f'  Mean score  : {np.mean(all_rr):.4f}')
    print(f'  Std score   : {np.std(all_rr):.4f}')
    print(f'  Min score   : {np.min(all_rr):.4f}')
    print(f'  Max score   : {np.max(all_rr):.4f}')
    if rr_spreads:
        print(f'  Avg spread (max-min per sampel): {np.mean(rr_spreads):.4f}')

print('\nContoh 10 sampel (reranker scores top-5):')
print('=' * 70)
for r in cr_p1[:10]:
    rr_scores = r.get('reranker_scores', [])
    bm_scores = r.get('retrieval_scores', [])
    status    = 'BENAR' if r['is_correct'] else 'SALAH'
    print(f'[{r["idx"]:3d}] GT={r["ground_truth"]} | Pred={r["predicted_label"]} | {status}')
    print(f'       Q: {r["question"][:70]}')
    if rr_scores:
        print(f'       Reranker: {[f"{s:.3f}" for s in rr_scores]}')
    if bm_scores:
        print(f'       BM25   : {[f"{s:.1f}" for s in bm_scores]}')
    print()